In [1]:
import pandas as pd

df=pd.read_csv("../data/processed/reduced/Final_data.csv")

In [2]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss

num = ['target_runs', 'runs_left', 'CRR', 'balls_left', 'RRR',
       'wickets_left']

X, y, groups = df[num], df['won'], df['match_id']

# same match-grouped split as the logistic regression
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
                           .split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

# SVMs are distance-based, so scale the features (fit on train only)
scaler = StandardScaler().fit(X_train)
Xtr, Xte = scaler.transform(X_train), scaler.transform(X_test)

def report(name, p):
    print(f'{name:12s} acc {accuracy_score(y_test, p > 0.5):.4f} | '
          f'auc {roc_auc_score(y_test, p):.4f} | '
          f'log loss {log_loss(y_test, p):.4f} | brier {brier_score_loss(y_test, p):.4f}')

# ---- 1. Linear SVM + probability calibration ----
inner_cv = list(GroupKFold(n_splits=3).split(Xtr, y_train, groups_train))   # grouped by match
linear_svm = CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=5000),
                                    method='sigmoid', cv=inner_cv)
linear_svm.fit(Xtr, y_train)
report('Linear SVM', linear_svm.predict_proba(Xte)[:, 1])

# ---- 2. RBF SVM (non-linear), trained on a sample because it is slow ----
rng = np.random.RandomState(0)
sub = rng.choice(len(Xtr), size=15000, replace=False)
rbf_svm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=0)
rbf_svm.fit(Xtr[sub], y_train.values[sub])
report('RBF SVM', rbf_svm.predict_proba(Xte)[:, 1])

Linear SVM   acc 0.8031 | auc 0.8969 | log loss 0.4035 | brier 0.1319


c:\Users\Lenovo\Documents\IPL ML\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


RBF SVM      acc 0.8019 | auc 0.8901 | log loss 0.4308 | brier 0.1382
